In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.32.2"

In [ ]:
%%writefile submission.py
"""Kaggriculture agent - generated by kagri/build_submission.py. Do not edit."""
import math
import time


CROPS = {
    "WHEAT":      {"seed": 10, "first_yield_day": 2, "max_yield_day": 4, "interval": 0, "max_yield": 6, "ongoing": False},
    "CARROT":     {"seed": 20, "first_yield_day": 2, "max_yield_day": 3, "interval": 0, "max_yield": 4, "ongoing": False},
    "TOMATO":     {"seed": 50, "first_yield_day": 8, "max_yield_day": 8, "interval": 1, "max_yield": 4, "ongoing": True},
    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},
    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},
}
ANIMALS = {
    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_held": 4, "product": "EGG"},
    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},
    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},
}
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
MARKET_PARAMS = {
    "WHEAT":      {"base":  25, "I0": 10000, "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",    "above_target": 0.20},
    "CARROT":     {"base":  35, "I0": 10000, "T": 450, "below_func": "log",    "below_target": 0.20, "above_func": "sqrt",   "above_target": 0.70},
    "TOMATO":     {"base":  60, "I0": 10000, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "sqrt",   "above_target": 0.60},
    "STRAWBERRY": {"base": 120, "I0": 10000, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
    "MELON":      {"base": 250, "I0": 10000, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.60},
    "EGG":        {"base":  50, "I0": 10000, "T": 332, "below_func": "linear", "below_target": 0.40, "above_func": "log",    "above_target": 0.20},
    "MILK":       {"base": 160, "I0": 10000, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
    "WOOL":       {"base": 200, "I0": 10000, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.20},
    "FERTILIZER": {"base": 100, "I0": 10000, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},
}
SHOPS = {
    "BAKERY":         ("EGG", "WHEAT"),
    "PIZZA_SHOP":     ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT":    ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE":     ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE":       ("CARROT",),
    "SMOOTHIE_SHOP":  ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}
LAND_PRICES = [1000, 2000, 4000]
ANIMAL_OF_PRODUCT = {v["product"]: k for k, v in ANIMALS.items()}


def _shape(f, x):
    x = max(0.0, x)
    if f == "linear": return x
    if f == "sq":     return x * x
    if f == "sqrt":   return math.sqrt(x)
    if f == "log":    return math.log(1.0 + x)
    if f == "log10":  return math.log10(1.0 + x)
    return x


def market_price(item, inv):
    p = MARKET_PARAMS[item]
    base, I0, T = p["base"], p["I0"], p["T"]
    if inv < I0:
        amp = p["below_target"] * base / _shape(p["below_func"], T)
        price = base + amp * _shape(p["below_func"], I0 - inv)
    else:
        amp = p["above_target"] * base / _shape(p["above_func"], T)
        price = base - amp * _shape(p["above_func"], inv - I0)
    return max(1, int(round(price)))


def avg_sale_price(item, inv, n):
    """Mean price realised selling n units starting from market inventory inv."""
    if n <= 0:
        return market_price(item, inv)
    step = max(1, n // 10)
    tot = cnt = 0
    i = 0
    while i < n:
        tot += market_price(item, inv + i)
        cnt += 1
        i += step
    return tot / cnt


DEFAULT_PARAMS = {
    # task priorities
    "pri_feed": 100.0,
    "pri_water_urgent": 95.0,
    "pri_water_bonus": 70.0,
    "pri_water_normal": 40.0,
    "pri_harvest_full": 105.0,
    "pri_harvest_ready": 72.0,
    "pri_care": 78.0,
    "pri_collect_fert": 84.0,
    "pri_fertilize": 92.0,
    "pri_plant": 65.0,
    "pri_build": 62.0,
    "pri_place_animal": 85.0,
    "pri_dig_weed": 25.0,
    "pri_drop": 72.0,
    "pri_pickup": 82.0,
    "dist_penalty": 6.0,
    # At/above this priority a task may pull a distant unit (task-centric);
    # below it, units pick their own nearest work (travel-efficient but
    # myopic). 0 = fully task-centric, which measured best; CEM tunes it.
    "critical_pri": 0.0,
    "match_mode": 1.0,          # >=0.5 = global best-pair matching, else legacy
    "match_cap": 90.0,          # tasks considered per turn (cost control)
    # economy
    "reserve_frac": 0.45,
    "pipeline_weight": 1.0,
    "absorb_weight": 1.0,
    "fert_keep": 0.0,
    "late_fill_days": 13.0,
    "prebuild_cap": 3.0,
    "backlog_weight": 1.0,
    "use_forecast": 1.0,        # 1 = best-response allocation from the market forecast
    "fert_credit": 1.0,
    "capital_rate": 0.10,       # daily shadow return on reinvested cash
    "reinvest_until": 18.0,     # after this day cash no longer compounds         # weight on fertiliser revenue when valuing livestock
    "shop_unlock_interval": 3.0,      # task-backlog -> crew-size scaling
    "save_ratio": 1.35,
    "save_reach": 0.5,          # only save for what we are already close to affording
    "save_cheap_cost": 0.0,    # seed cost still allowed while saving         # save for a better tile use rather than substitute
    "save_until_days": 8.0,     # ... but never in the endgame        # animal structures built ahead of owning the animal     # days-left below which spare land gets filler crops          # fertiliser held back for our own crops       # how much town demand offsets our own supply     # how strongly committed output suppresses new planting
    "endgame_dump_day": 27.0,
    "liquidate_hour": 14.0,     # hour on the last day to start running stock home
    "land_day1": 7.0,          # earliest day to buy quadrant 1
    "land_day2": 10.0,
    "land_day3": 99.0,
    "land_cash_buffer": 0.0,
    "land_empty_max": 60.0,     # only buy land if fewer than this many empties
    "max_hands": 16.0,
    "useful_acts": 8.0,         # effective productive actions per unit per day
    "wheat_carry": 8.0,
    "wheat_min_carry": 3.0,
    "carry_mult": 1.0,          # multiplier on the per-unit feeding share
    "fert_carry": 3.0,
    "runner_slack": 99.0,        # extra feed-runners beyond the strict minimum
    "fetch_mode": 1.0,          # 1 = fetch only when assigned a task needing it
    "fert_runners": 99.0,        # units fetching fertiliser per turn          # fertiliser taken per shed trip
    "wheat_buffer_days": 2.0,
    "drop_threshold": 7.0,
    "drop_threshold_panic": 3.0,
    "opp_supply_weight": 0.6,   # how much opponent pending supply front-runs us
    "max_sell_per_turn": 8.0,   # meter sales so we don't walk our own price down
    "sell_slots": 4.0,          # market-order slots reserved for selling
    "sell_slots_full": 7.0,     # ... when the shed is close to overflowing
    "shed_panic_frac": 0.72,    # shed fullness above which we dump at any price
    # portfolio
    "max_goose": 0.0,
    "max_cow": 8.0,
    "max_sheep": 6.0,
    "max_melon": 26.0,
    "max_strawberry": 40.0,
    "max_tomato": 0.0,
    "max_wheat": 11.0,
    "max_carrot": 0.0,
    "animal_invest_day": 18.0,
    "goose_cash_buffer": 200.0,
    # capital / cash-flow discipline
    "reserve_days": 0.0,        # days of burn to keep in the bank
    "min_reserve": 244.0,       # absolute cash floor
    "cash_urgency": 0.0,      # penalty per day of payback when cash is tight
    "comfort_cash": 2500.0,     # above this, long-payback crops are fine
    "acts_per_crop": 2.6,       # action-budget cost of servicing a crop tile
    "acts_per_animal": 5.2,     # ... an animal tile
    "invest_frac": 1.0,        # max fraction of free cash to deploy per turn
    "hire_cash_frac": 0.12,     # slice of cash usable for hiring
    "hire_min_budget": 60.0,    # always affordable floor for hiring
    "wheat_hysteresis": 12.0,   # dead-band so we never buy and sell wheat together
    "weed_boost_idle": 45.0,    # dig priority bonus when labour is idle
    "hire_window": 2.0,         # only hire during the first N hours of a day
    "hires_per_turn": 4.0,      # HIRE orders per turn (10-slot queue is scarce)
    "feed_cost": 22.0,          # imputed cost of one wheat unit of feed
    "wheat_per_animal": 0.9,    # wheat tiles to keep per animal
    "w_goose": 1.0,            # valuation multipliers, tuned by CEM
    "w_melon": 1.0,
    "w_wheat": 1.0,
    "w_carrot": 1.0,
    "w_tomato": 1.0,
    "w_straw": 1.0,
    "w_cow": 1.0,
    "w_sheep": 1.0,
}


def _Forecast(obs, me, p):
    """Resolved lazily so the class can live either in kagri.forecast (dev) or
    inlined below it in the generated single-file submission."""
    try:
        from kagri.forecast import Forecast as F
    except Exception:
        F = globals()["Forecast"]
    return F(obs, me, p)


class Agent:
    def __init__(self, params=None):
        self.p = dict(DEFAULT_PARAMS)
        if params:
            self.p.update(params)
        self.board = 10
        # Per-player caches. Kaggle's upload validation runs main.py against
        # ITSELF, and the harness may share one module between both seats, so
        # this state is keyed by player id rather than held as a single dict.
        self._task_backlog = 0.0   # real work measured last turn, drives hiring
        self._targets = {}     # player -> {unit index: (x, y) it is walking to}
        self._target = {}

    # ---------------------------------------------------------------- helpers
    def shed_tiles(self):
        h = self.board // 2
        return [(h - 1, h - 1), (h, h - 1), (h - 1, h), (h, h)]

    def open_shed_tiles(self, tiles):
        out = [t for t in self.shed_tiles() if tiles[t[1]][t[0]] != "LOCKED"]
        return out or [self.shed_tiles()[0]]

    @staticmethod
    def dist(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    @staticmethod
    def step_towards(src, dst):
        dx, dy = dst[0] - src[0], dst[1] - src[1]
        if dx > 0: return ["EAST"]
        if dx < 0: return ["WEST"]
        if dy > 0: return ["SOUTH"]
        if dy < 0: return ["NORTH"]
        return None

    # ------------------------------------------------------------- valuation
    def crop_plan_value(self, crop, days_left, minv, planned):
        """Coins per tile-day from planting `crop` now, net of seed and of the
        price impact of our own added supply."""
        cd = CROPS[crop]
        if not cd["ongoing"]:
            myd, fyd = cd["max_yield_day"], cd["first_yield_day"]
            ws = (myd + 1) // 2
            if days_left >= myd + 1:
                units = min(cd["max_yield"], 1 + (myd - ws + 1))
                need = myd
            elif days_left >= fyd + 1:
                d = int(days_left) - 1
                units = min(cd["max_yield"], 1 + max(0, d - ws + 1))
                need = d
            else:
                return -1e9
        else:
            fyd, iv, my = cd["first_yield_day"], cd["interval"], cd["max_yield"]
            if days_left < fyd + 2:
                return -1e9
            n = 1 + int((days_left - 1 - fyd) // max(1, iv))
            units = min(my, n)
            need = fyd + iv * (units - 1)
            # Fertilised ongoing crops yield 2 per production instead of 1, and
            # every animal produces a free fertiliser every day - so with a herd
            # on the farm strawberry/tomato are worth double.
            if self._fert_supply > 0:
                units *= 2
        if units <= 0:
            return -1e9
        absorbed = self.town_absorption(crop, need, self._day) * self.p["absorb_weight"]
        net = max(0.0, planned.get(crop, 0) - absorbed)
        inv = minv.get(crop, 10000) + net
        price = avg_sale_price(crop, inv, units)
        return (units * price - cd["seed"]) / max(1.0, need)

    def animal_plan_value(self, animal, days_left, minv, planned):
        ad = ANIMALS[animal]
        if days_left < ad["first_yield_day"] + 3:
            return -1e9
        iv = ad["interval"]
        n_prod = 1 + int((days_left - 1 - ad["first_yield_day"]) // iv)
        if n_prod <= 0:
            return -1e9
        # with daily CARE the bank pays out ~interval extra per production,
        # capped by max_held
        per = min(ad["max_held"], 1 + iv)
        units = n_prod * per
        prod = ad["product"]
        absorbed = self.town_absorption(prod, days_left, self._day) * self.p["absorb_weight"]
        net = max(0.0, planned.get(prod, 0) - absorbed)
        inv = minv.get(prod, 10000) + net
        price = avg_sale_price(prod, inv, units)
        # Value feed at what it costs us to *produce* a wheat unit, not at the
        # market price - buying feed drives wheat's scarcity price up steeply
        # and would make every animal look unprofitable.
        feed_cost = days_left * self.p["feed_cost"]
        gross = units * price - ad["cost"] - feed_cost
        return gross / max(1.0, days_left)

    # ---------------------------------------------------------------- entry
    def __call__(self, obs):
        try:
            return self.act(obs)
        except Exception:
            return {"farmer": ["PASS"], "hands": [], "market": []}

    def act(self, obs):
        p = self.p
        me = obs["player"]
        farm = obs["farms"][me]
        priv = obs["private"]
        tiles = farm["tiles"]
        self.board = len(tiles)
        day, hour = obs["day"], obs["hour"]
        days_left = 30 - day
        prices = obs["market"]["prices"]
        minv = obs["market"]["inventory"]
        shed = priv["shed"]
        money = farm["money"]

        units = [list(farm["farmer"])] + [list(h) for h in farm["hands"]]
        invs = list(priv["inventories"])
        while len(invs) < len(units):
            invs.append({})

        c = self.census(tiles)
        c["n_animals"] = sum(c["animals"].values())
        self._forecast = None
        self._plan_cache = None
        if p["use_forecast"] >= 0.5:
            try:
                self._forecast = _Forecast(obs, me, p)
            except Exception:
                self._forecast = None
        self._target = self._targets.setdefault(me, {})
        if hour == 0:
            self._target.clear()
        self._money = money
        carried = sum(sum(i.values()) for i in invs)
        self._shed_pressure = (sum(shed.values()) + carried) / 100.0
        self._shed_snapshot = dict(shed)
        self._opp_supply = self.opponent_supply(obs, me, days_left)
        self._carried_wheat = sum(i.get("WHEAT", 0) for i in invs)
        self._carried = {}
        for i in invs:
            for k, v in i.items():
                self._carried[k] = self._carried.get(k, 0) + v
        self._shops = list(obs.get("town", {}).get("unlocked_shops", []) or [])
        self._fert_supply = c["n_animals"] + shed.get("FERTILIZER", 0)
        self._day = day
        self._liquidate = (days_left <= 1 and hour >= p["liquidate_hour"])
        self._seeds = dict(priv["seeds"])
        self._n_units = len(units)
        # spare labour -> clearing weeds becomes worth an action
        demand = (sum(c["crops"].values()) * p["acts_per_crop"]
                  + c["n_animals"] * p["acts_per_animal"])
        supply = 24.0 * len(units)
        self._weed_boost = p["weed_boost_idle"] if demand < supply * 0.75 else 0.0

        target = self.target_composition(days_left, minv, c)
        market = self.market_orders(obs, farm, priv, day, hour, days_left,
                                    prices, minv, shed, money, c, target)
        tasks = self.build_tasks(obs, tiles, day, days_left, prices, shed,
                                 c, target, invs, minv)
        self._task_backlog = len(tasks)
        acts = self.assign(tasks, units, invs, tiles, shed, c)
        return {"farmer": acts[0], "hands": acts[1:], "market": market}

    # ---------------------------------------------------------------- census
    def census(self, tiles):
        animals = {"GOOSE": 0, "COW": 0, "SHEEP": 0}
        crops = {k: 0 for k in CROPS}
        empty, weeds, free_coop, free_past = [], [], [], []
        for y in range(self.board):
            row = tiles[y]
            for x in range(self.board):
                t = row[x]
                if t is None:
                    empty.append((x, y))
                elif t == "LOCKED":
                    continue
                else:
                    k = t.get("kind")
                    if k == "PLANT":
                        crops[t["crop"]] += 1
                    elif k == "WEED":
                        weeds.append((x, y))
                    elif k == "COOP":
                        if t.get("animal"):
                            animals["GOOSE"] += 1
                        else:
                            free_coop.append((x, y))
                    elif k == "PASTURE":
                        a = t.get("animal")
                        if a:
                            animals[a] += 1
                        else:
                            free_past.append((x, y))
        return {"animals": animals, "crops": crops, "empty": empty, "weeds": weeds,
                "free_coop": free_coop, "free_past": free_past}

    # --------------------------------------------------- target composition
    def target_composition(self, days_left, minv, c):
        """How many of each tile type we want, given the remaining horizon."""
        p = self.p
        t = {}
        t["GOOSE"] = int(p["max_goose"]) if days_left >= ANIMALS["GOOSE"]["first_yield_day"] + 3 else 0
        t["COW"] = int(p["max_cow"]) if days_left >= ANIMALS["COW"]["first_yield_day"] + 3 else 0
        t["SHEEP"] = int(p["max_sheep"]) if days_left >= ANIMALS["SHEEP"]["first_yield_day"] + 3 else 0
        t["MELON"] = int(p["max_melon"]) if days_left >= 13 else 0
        t["STRAWBERRY"] = int(p["max_strawberry"]) if days_left >= 12 else 0
        t["TOMATO"] = int(p["max_tomato"]) if days_left >= 10 else 0
        t["WHEAT"] = int(max(p["max_wheat"],
                             c["n_animals"] * p["wheat_per_animal"]))
        # Once strawberry/melon can no longer finish, their tiles free up and
        # nothing in the normal portfolio is allowed to take them. Wheat and
        # carrot are short-cycle and have deep town demand, so let them fill
        # whatever is left rather than idling the land.
        if days_left <= p["late_fill_days"]:
            free = len(c["empty"]) + 4
            t["WHEAT"] = max(t["WHEAT"], int(p["max_wheat"] + free))
            t["CARROT"] = max(t.get("CARROT", 0), int(p["max_carrot"] + free))
        t["CARROT"] = int(p["max_carrot"])
        return t

    # ------------------------------------------------------ capital & labour
    def hire_cum_cost(self, n):
        cost, fa, fb = 0, 1, 1
        for _ in range(int(n)):
            cost += fa
            fa, fb = fb, fa + fb
        return cost

    def affordable_hands(self, money):
        """Most hands we can sustain for a day using a slice of current cash."""
        p = self.p
        budget = max(0.0, money) * 0.20
        n, cost, fa, fb = 0, 0, 1, 1
        while n < int(p["max_hands"]):
            if cost + fa > budget:
                break
            cost += fa
            fa, fb = fb, fa + fb
            n += 1
        return n

    def daily_burn(self, money, n_animals, wheat_price):
        n = self.affordable_hands(money)
        return self.hire_cum_cost(n) + n_animals * max(1, wheat_price)

    def cash_floor(self, money, n_animals, wheat_price):
        p = self.p
        return max(p["min_reserve"],
                   self.daily_burn(money, n_animals, wheat_price) * p["reserve_days"])

    def service_capacity(self, money, c):
        """How many tiles our labour force can actually keep alive."""
        p = self.p
        hands = self.affordable_hands(money)
        budget = 24.0 * (1 + hands)
        used = (sum(c["crops"].values()) * p["acts_per_crop"]
                + c["n_animals"] * p["acts_per_animal"])
        return max(0.0, budget - used)

    def urgency(self, money):
        p = self.p
        return p["cash_urgency"] * max(0.0, 1.0 - money / max(1.0, p["comfort_cash"]))

    # ------------------------------------------------------------- filling
    def choose_fill(self, days_left, minv, c, planned, pending, money):
        """Pick the best type for one more empty tile, or None."""
        p = self.p
        w = {"GOOSE": p["w_goose"], "COW": p["w_cow"], "SHEEP": p["w_sheep"],
             "MELON": p["w_melon"], "WHEAT": p["w_wheat"], "CARROT": p["w_carrot"],
             "TOMATO": p["w_tomato"], "STRAWBERRY": p["w_straw"]}
        target = self.target_composition(days_left, minv, c)
        u = self.urgency(money)
        best, best_v = None, 0.0
        # Value of the best option we CANNOT currently afford. If that is much
        # better than anything we can afford, spending the cash on the cheap
        # substitute is a trap: we never accumulate enough for the good one.
        # Livestock is the whole early engine, so saving for it beats buying
        # another melon seed today.
        blocked_v = 0.0
        for a in ANIMALS:
            have = (c["animals"][a] + pending.get(a, 0)
                    + self._shed_snapshot.get(a, 0) + self._carried.get(a, 0))
            if have >= target.get(a, 0):
                continue
            v = self.animal_plan_value(a, days_left, minv, planned) * w[a]
            v /= (1.0 + u * ANIMALS[a]["first_yield_day"])
            if ANIMALS[a]["cost"] > money:
                # Only worth saving for something we can nearly afford. If we
                # are far short, holding out blocks every cheaper purchase too
                # and we end the season having bought nothing.
                if money >= ANIMALS[a]["cost"] * p["save_reach"]:
                    blocked_v = max(blocked_v, v)
                continue
            if v > best_v:
                best_v, best = v, ("ANIMAL", a)
        for cr in CROPS:
            have = c["crops"][cr] + pending.get(cr, 0)
            if have >= target.get(cr, 0):
                continue
            v = self.crop_plan_value(cr, days_left, minv, planned) * w[cr]
            v /= (1.0 + u * CROPS[cr]["first_yield_day"])
            if CROPS[cr]["seed"] > money:
                if money >= CROPS[cr]["seed"] * p["save_reach"]:
                    blocked_v = max(blocked_v, v)
                continue
            if v > best_v:
                best_v, best = v, ("CROP", cr)
        if blocked_v > best_v * p["save_ratio"] and days_left > p["save_until_days"]:
            # Saving for livestock, but idling land is its own waste. A $10
            # wheat seed barely delays a $400 cow, so cheap filler is still
            # allowed - only the expensive substitutes are blocked.
            cheap, cheap_v = None, 0.0
            for cr in CROPS:
                if CROPS[cr]["seed"] > p["save_cheap_cost"] or CROPS[cr]["seed"] > money:
                    continue
                if c["crops"][cr] + pending.get(cr, 0) >= target.get(cr, 0):
                    continue
                v = self.crop_plan_value(cr, days_left, minv, planned) * w[cr]
                if v > cheap_v:
                    cheap_v, cheap = v, ("CROP", cr)
            return cheap, cheap_v
        return best, best_v

    def opponent_supply(self, obs, me, days_left):
        """What the opponent is about to dump on the market.

        Their farm grid is public: we can read yield_units already sitting on
        their tiles plus what their crops/animals will produce. Since we share
        one price curve, whoever sells first gets the better price - so if they
        are sitting on 60 harvest-ready melons we want to be out before them.
        """
        sup = {}
        farms = obs["farms"]
        if len(farms) < 2:
            return sup
        opp = farms[1 - me]
        for row in opp["tiles"]:
            for t in row:
                if not isinstance(t, dict):
                    continue
                k = t.get("kind")
                yu = t.get("yield_units", 0) or 0
                if k == "PLANT":
                    cr = t["crop"]
                    sup[cr] = sup.get(cr, 0) + yu
                    if CROPS[cr]["ongoing"] and days_left > 1:
                        sup[cr] = sup.get(cr, 0) + min(
                            CROPS[cr]["max_yield"],
                            int(days_left // max(1, CROPS[cr]["interval"])))
                elif k in ("COOP", "PASTURE") and t.get("animal"):
                    a = t["animal"]
                    prod = ANIMALS[a]["product"]
                    iv = ANIMALS[a]["interval"]
                    sup[prod] = sup.get(prod, 0) + yu + int(days_left // max(1, iv))
        return sup

    def town_absorption(self, item, days, day):
        """Units of `item` the town will consume over the next `days` days.

        This is the demand side of the price curve and we were ignoring it.
        Every unlocked shop eats 1 of each product it wants every
        townShopSellInterval turns (6/day; 12/day for single-product shops),
        and the town centre eats 1 of everything every 12 turns, scaling to 2x
        after day 10 and 4x after day 20. That consumption pulls inventory back
        down, which is what lets us keep selling near base price - so a product
        several shops demand can absorb far more of our output than the static
        glut curve alone suggests.
        """
        if item == "FERTILIZER":
            per_day = 0.0
        else:
            mult = 4 if day >= 20 else (2 if day >= 10 else 1)
            per_day = 2.0 * mult          # town centre: 24/12 ticks per day
        for shop in self._shops:
            prods = SHOPS.get(shop, ())
            if item in prods:
                per_day += 12.0 if len(prods) == 1 else 6.0
        return per_day * max(0.0, days)

    def pipeline_supply(self, c, days_left):
        """Units of each product we are already committed to producing.

        Without this the planner happily plants a 20th melon while 100 unsold
        melons sit in the shed and the melon price is on the floor: each new
        tile is valued as if the market were untouched.
        """
        sup = dict(self._shed_snapshot)
        for cr, n in c["crops"].items():
            if n:
                sup[cr] = sup.get(cr, 0) + n * CROPS[cr]["max_yield"]
        for a, n in c["animals"].items():
            if n:
                ad = ANIMALS[a]
                iv = ad["interval"]
                n_prod = max(0, int((days_left - ad["first_yield_day"]) // iv))
                per = min(ad["max_held"], 1 + iv)
                sup[ad["product"]] = sup.get(ad["product"], 0) + n * n_prod * per
        return sup

    def fill_plan(self, days_left, minv, c, n_slots, money):
        """Plan the next n_slots empty tiles, respecting labour and cash."""
        p = self.p
        if getattr(self, "_forecast", None) is not None:
            key = (n_slots, int(money))
            if self._plan_cache and self._plan_cache[0] == key:
                return self._plan_cache[1]
            cap = self.service_capacity(money, c)
            budget = max(0.0, money - self.cash_floor(
                money, c["n_animals"], market_price("WHEAT", minv.get("WHEAT", 10000))))
            budget *= p["invest_frac"]
            try:
                out = self._forecast.allocate(n_slots, budget, cap, c, days_left)
                self._plan_cache = (key, out)
                return out
            except Exception:
                pass
        planned = {k: v * p["pipeline_weight"]
                   for k, v in self.pipeline_supply(c, days_left).items()}
        pending = {}
        out = []
        cap = self.service_capacity(money, c)
        budget = max(0.0, money - self.cash_floor(money, c["n_animals"],
                                                  market_price("WHEAT", minv.get("WHEAT", 10000))))
        budget *= p["invest_frac"]
        for _ in range(n_slots):
            best, v = self.choose_fill(days_left, minv, c, planned, pending, budget)
            if best is None:
                break
            kind, name = best
            cost = ANIMALS[name]["cost"] if kind == "ANIMAL" else CROPS[name]["seed"]
            acts = p["acts_per_animal"] if kind == "ANIMAL" else p["acts_per_crop"]
            if cost > budget or acts > cap:
                break
            budget -= cost
            cap -= acts
            pending[name] = pending.get(name, 0) + 1
            if kind == "CROP":
                planned[name] = planned.get(name, 0) + CROPS[name]["max_yield"]
            else:
                planned[ANIMALS[name]["product"]] = planned.get(ANIMALS[name]["product"], 0) + int(days_left)
            out.append(best)
        return out

    def pack_orders(self, B, hour, shed, c):
        """Fit orders into maxMarketOrdersPerTurn (10) slots.

        The cap is a real constraint: naively emitting every HIRE first starves
        selling and seed-buying. We therefore reserve slots per category and
        spread hiring across the early hours of the day.
        """
        p = self.p
        slots = 10
        out = []
        shed_full = self._shed_pressure > p["shed_panic_frac"]

        # 1. selling always gets first refusal - it funds everything else,
        #    and a full shed silently discards production.
        n_sell = min(len(B["sell"]), int(p["sell_slots_full"]) if shed_full
                     else int(p["sell_slots"]))
        out.extend(B["sell"][:n_sell])

        # 2. feed: losing an animal costs far more than a wheat order
        out.extend(B["feed"][:1])

        # 3. hiring, rate-limited so it cannot monopolise the queue
        per_turn = int(p["hires_per_turn"])
        out.extend(B["hire"][:per_turn])

        # 4. the rest, best-value first
        for key in ("seed", "animal", "land"):
            for o in B[key]:
                if len(out) >= slots:
                    break
                out.append(o)
        # 5. backfill with any remaining sells / hires
        for key in ("sell", "hire"):
            for o in B[key]:
                if len(out) >= slots:
                    break
                if o not in out:
                    out.append(o)
        return out[:slots]

    # ---------------------------------------------------------------- market
    def market_orders(self, obs, farm, priv, day, hour, days_left, prices,
                      minv, shed, money, c, target):
        p = self.p
        B = {"sell": [], "hire": [], "feed": [], "seed": [], "animal": [], "land": []}
        endgame = day >= p["endgame_dump_day"]
        final_dump = (days_left <= 1 and hour >= p["liquidate_hour"] + 2)
        n_animals = c["n_animals"]

        wheat_price = market_price("WHEAT", minv["WHEAT"])
        floor = self.cash_floor(money, n_animals, wheat_price)

        # ---------- 1. HIRE (cheapest, highest leverage) -------------------
        # Labour is extremely cheap (10 hands = ~$143/day), so we hire up to
        # the workload every day, and top up later in the day once sales land.
        if hour <= int(p["hire_window"]):
            # Size the crew off the REAL task backlog measured last turn, not
            # off tile counts. Empty tiles we are deliberately not planting
            # generate no work, so counting them over-hires early (measured
            # 41-75% idle on days 0-11); and once the farm matures the backlog
            # explodes and we were leaving fertilise/care undone.
            workload = max(self._task_backlog * p["backlog_weight"],
                           sum(c["crops"].values()) * p["acts_per_crop"]
                           + n_animals * p["acts_per_animal"]
                           + len(c["weeds"]) * 0.8)
            # A unit does NOT get 24 useful actions a day - most of its turns
            # go on travel. Sizing the crew off 24 chronically under-hires;
            # `useful_acts` is the measured effective throughput per unit.
            want = min(int(p["max_hands"]),
                       int(math.ceil(workload / max(1.0, p["useful_acts"]))))
            already = farm["hires_today"]
            # Hire out of surplus, not out of capital. Early on there is no
            # income yet, and a big crew hired on day 0 eats the seed money
            # that the whole compounding engine depends on.
            cap = max(p["hire_min_budget"],
                      max(0.0, money - floor) * p["hire_cash_frac"])
            cost, fa, fb, n = 0, 1, 1, 0
            for _ in range(already):          # advance the fib cursor
                fa, fb = fb, fa + fb
            while already + n < want:
                if cost + fa > cap:
                    break
                cost += fa
                fa, fb = fb, fa + fb
                n += 1
            for _ in range(n):
                B["hire"].append(["HIRE"])
            money -= cost

        # cash available for *investment* (seeds, animals, land)
        invest = max(0.0, money - floor) * p["invest_frac"]

        # ---------- 2. SELL ------------------------------------------------
        sells = []
        for item in PRODUCTS:
            n = shed.get(item, 0)
            if n <= 0:
                continue
            if item == "WHEAT" and not final_dump:
                # keep enough for feed AND for what the units will carry, plus
                # a hysteresis band so we never buy and sell in the same phase
                keep = (int(n_animals * p["wheat_buffer_days"])
                        + int(p["wheat_carry"]) * int(p["max_hands"])
                        + int(p["wheat_hysteresis"]))
                n -= keep
            if item == "FERTILIZER" and not endgame:
                # Animals make one free fertiliser each per day. Keep enough to
                # keep the ongoing crops fertilised and sell the rest - it is a
                # $100-base product and the surplus is pure profit.
                n -= int(p["fert_keep"])
            if n <= 0:
                continue
            if final_dump:
                qty = n
            else:
                reserve = MARKET_PARAMS[item]["base"] * p["reserve_frac"]
                if endgame:
                    reserve = 1
                # The shed holds only `shedCapacity` items and end-of-day
                # overflow is destroyed outright. Once it fills up, holding out
                # for a better price is strictly worse than selling cheap.
                if self._shed_pressure > p["shed_panic_frac"]:
                    reserve = 1
                # Price the sale against a market that also absorbs the
                # opponent's imminent supply: if they are about to flood this
                # good, waiting for a better price is a losing bet.
                cur = minv[item] + int(self._opp_supply.get(item, 0)
                                       * p["opp_supply_weight"])
                qty = 0
                for _ in range(int(n)):
                    pr = market_price(item, cur)
                    if pr <= 1 or pr < reserve:
                        break
                    qty += 1
                    cur += 1
                # Price is quoted per unit as our own supply lands, so dumping
                # 50 units walks the price down 50 steps in one turn. The town
                # drains inventory every few turns, which lifts the price back
                # up - so metering sales out over the day realises far more.
                if not endgame and self._shed_pressure <= p["shed_panic_frac"]:
                    qty = min(qty, int(p["max_sell_per_turn"]))
            if qty > 0:
                sells.append((prices.get(item, 0) * qty, ["SELL", item, int(qty)]))
        sells.sort(reverse=True, key=lambda s: s[0])
        B["sell"] = [o for _, o in sells[:6]]

        if final_dump:
            return B["sell"][:10]

        # ---------- 3. LAND ------------------------------------------------
        n_extra = len(farm["unlocked_quadrants"]) - 1
        if n_extra < 3:
            gate = [p["land_day1"], p["land_day2"], p["land_day3"]][n_extra]
            cost = LAND_PRICES[n_extra]
            if (day >= gate and invest >= cost + p["land_cash_buffer"]
                    and len(c["empty"]) <= p["land_empty_max"] and days_left >= 8):
                B["land"].append(["BUY_LAND"])
                invest -= cost

        # ---------- 4. ANIMALS (buy only what we have a home for) ----------
        if day <= p["animal_invest_day"]:
            slots = {"COOP": len(c["free_coop"]), "PASTURE": len(c["free_past"])}
            # plus structures we are about to build on empty tiles
            plan = self.fill_plan(days_left, minv, c, min(len(c["empty"]), 12), invest)
            for kind, name in plan:
                if kind == "ANIMAL":
                    slots[ANIMALS[name]["structure"]] += 1
            for a in ("GOOSE", "SHEEP", "COW"):
                ad = ANIMALS[a]
                if days_left < ad["first_yield_day"] + 3:
                    continue
                have = c["animals"][a] + shed.get(a, 0) + self._carried.get(a, 0)
                room = target.get(a, 0) - have
                room = min(room, slots.get(ad["structure"], 0))
                if room <= 0:
                    continue
                afford = int(max(0, invest - p["goose_cash_buffer"]) // ad["cost"])
                buy = min(room, afford, 6)
                if buy > 0:
                    B["animal"].append(["BUY_ANIMAL", a, buy])
                    invest -= buy * ad["cost"]
                    slots[ad["structure"]] -= buy

        # ---------- 5. SEEDS -----------------------------------------------
        seeds = priv["seeds"]
        if c["empty"] and days_left > 1:
            plan = self.fill_plan(days_left, minv, c, min(len(c["empty"]), 20), invest)
            need = {}
            for kind, name in plan:
                if kind == "CROP":
                    need[name] = need.get(name, 0) + 1
            for cr, n in sorted(need.items(), key=lambda kv: -CROPS[kv[0]]["seed"]):
                buy = n - seeds.get(cr, 0)
                if buy <= 0:
                    continue
                cost = CROPS[cr]["seed"]
                buy = min(buy, int(invest // cost))
                if buy > 0:
                    B["seed"].append(["BUY_SEED", cr, buy])
                    invest -= buy * cost

        # ---------- 6. FEED WHEAT -------------------------------------------
        if n_animals > 0:
            need = int(n_animals * p["wheat_buffer_days"]) + 4
            have = shed.get("WHEAT", 0) + self._carried_wheat
            if have < need:
                pr = market_price("WHEAT", minv["WHEAT"])
                buy = min(need - have, int(money // max(1, pr)), 60)
                if buy > 0:
                    B["feed"].append(["BUY_PRODUCT", "WHEAT", int(buy)])

        return self.pack_orders(B, hour, shed, c)

    # ----------------------------------------------------------------- tasks
    def build_tasks(self, obs, tiles, day, days_left, prices, shed, c, target,
                    invs, minv):
        p = self.p
        tasks = []
        add = tasks.append
        endgame = day >= p["endgame_dump_day"]
        last_day = days_left <= 1
        want_fert = c["crops"]["TOMATO"] + c["crops"]["STRAWBERRY"] > 0

        for y in range(self.board):
            row = tiles[y]
            for x in range(self.board):
                t = row[x]
                if t is None or t == "LOCKED":
                    continue
                k = t.get("kind")
                if k == "PLANT":
                    cd = CROPS[t["crop"]]
                    age = day - t["planted_day"]
                    val = prices.get(t["crop"], 1) / 100.0
                    yu = t.get("yield_units", 0)
                    ready = age >= cd["first_yield_day"]
                    if yu > 0 and ready:
                        if cd["ongoing"]:
                            # Held yield caps at max_yield, so anything the next
                            # scheduled production would push past the cap is
                            # destroyed. A fertilised strawberry makes 2/tick
                            # against a cap of 4 - wait for it to be "full" and
                            # you have already lost half the crop.
                            full = yu >= cd["max_yield"]
                            add((p["pri_harvest_full"] if full else p["pri_harvest_ready"], (x, y), ["HARVEST"], None))
                        elif age >= cd["max_yield_day"] or endgame or last_day:
                            add((p["pri_harvest_full"] + val, (x, y), ["HARVEST"], None))
                    if not t["watered_today"] and not last_day:
                        if t.get("consecutive_unwatered", 0) >= 1:
                            pri = p["pri_water_urgent"]
                        elif not cd["ongoing"]:
                            ws = (cd["max_yield_day"] + 1) // 2
                            pri = (p["pri_water_bonus"] + val) if ws <= age <= cd["max_yield_day"] else p["pri_water_normal"]
                        else:
                            pri = p["pri_water_bonus"] if age >= cd["first_yield_day"] - 1 else p["pri_water_normal"]
                        add((pri, (x, y), ["WATER"], None))
                    if t.get("fertilized_until_day", -1) < day and not endgame:
                        ok = False
                        if cd["ongoing"] and age >= cd["first_yield_day"] - 1:
                            ok = True
                        elif not cd["ongoing"]:
                            ws = (cd["max_yield_day"] + 1) // 2
                            ok = ws <= age <= cd["max_yield_day"] and yu < cd["max_yield"]
                        if ok:
                            add((p["pri_fertilize"] + val, (x, y), ["FERTILIZE"], "FERTILIZER"))
                elif k == "WEED":
                    if days_left > 2:
                        add((p["pri_dig_weed"] + self._weed_boost, (x, y), ["DIG"], None))
                elif k in ("COOP", "PASTURE"):
                    a = t.get("animal")
                    if a:
                        ad = ANIMALS[a]
                        if not t["fed_today"] and not last_day:
                            add((p["pri_feed"], (x, y), ["FEED"], "WHEAT"))
                        yu = t.get("yield_units", 0)
                        if yu > 0:
                            full = yu >= ad["max_held"]
                            add((p["pri_harvest_full"] if full else p["pri_harvest_ready"], (x, y), ["HARVEST"], None))
                        if not t["cared_today"] and days_left > 1:
                            add((p["pri_care"], (x, y), ["CARE"], None))
                        if t.get("fertilizer_available"):
                            add((p["pri_collect_fert"], (x, y), ["COLLECT_FERTILIZER"], None))
                    else:
                        want = "GOOSE" if k == "COOP" else ("SHEEP" if (shed.get("SHEEP", 0) or any(i.get("SHEEP", 0) for i in invs)) else "COW")
                        add((p["pri_place_animal"], (x, y), ["PLACE", want], want))

        # empty tiles -> plant seeds we already own, or build a home for an
        # animal we already own / can afford. Buying is the market's job.
        if c["empty"] and days_left > 1:
            sh = self.shed_tiles()[0]
            slots = sorted(c["empty"], key=lambda q: self.dist(q, sh))
            queue = []
            # seeds on hand, most valuable first
            for cr, n in sorted(self._seeds.items(),
                                key=lambda kv: -self.crop_plan_value(kv[0], days_left, minv, {})):
                if n <= 0 or self.crop_plan_value(cr, days_left, minv, {}) <= 0:
                    continue
                queue.extend([("CROP", cr)] * int(n))
            # structures for animals sitting in the shed / inventories
            for a in ANIMALS:
                held = shed.get(a, 0) + sum(i.get(a, 0) for i in invs)
                free = len(c["free_coop"]) if ANIMALS[a]["structure"] == "COOP" else len(c["free_past"])
                need = held - free
                if need > 0:
                    queue.extend([("ANIMAL", a)] * int(need))
            # if we still have room and cash, pre-build coops for future geese
            if len(queue) < len(slots):
                # Speculative structures only a few ahead of what we can afford:
                # building 11 pastures we have no animals for burns actions and
                # squats on land that a strawberry could be growing on.
                room = min(len(slots) - len(queue), int(p["prebuild_cap"]))
                extra = self.fill_plan(days_left, minv, c, room, self._money)
                queue.extend([e for e in extra if e[0] == "ANIMAL"][:room])
            for pos, (kind, name) in zip(slots, queue):
                if kind == "CROP":
                    add((p["pri_plant"], pos, ["PLANT", name], ("SEED", name)))
                else:
                    struct = ANIMALS[name]["structure"]
                    add((p["pri_build"], pos, ["BUILD_COOP" if struct == "COOP" else "BUILD_PASTURE"], None))
        return tasks

    # ------------------------------------------------------------- assignment
    def assign(self, tasks, units, invs, tiles, shed, c):
        p = self.p
        n = len(units)
        actions = [["PASS"] for _ in range(n)]
        used = [False] * n
        sheds = self.open_shed_tiles(tiles)

        # -- what do we need from the shed? --------------------------------
        self._unfed = sum(1 for t in tasks if t[2][0] == "FEED")
        need_wheat = self._unfed > 0 and shed.get("WHEAT", 0) > 0
        # Only send as many units to fetch feed as are actually needed to do
        # the feeding. Letting every unit top up meant ~13 shed trips a day for
        # 14 animals - roughly 400 actions a game that the strong agents spend
        # on watering and harvesting instead.
        wheat_runners = int(math.ceil(self._unfed / max(1.0, p["wheat_min_carry"])))
        wheat_runners = max(1, int(wheat_runners * p["runner_slack"]))
        fert_runners = max(1, int(p["fert_runners"]))
        need_fert = any(t[3] == "FERTILIZER" for t in tasks) and shed.get("FERTILIZER", 0) > 0
        animal_need = {}
        for t in tasks:
            if t[2][0] == "PLACE":
                a = t[2][1]
                if shed.get(a, 0) > 0:
                    animal_need[a] = animal_need.get(a, 0) + 1

        logistics = []
        for i in range(n):
            invi = invs[i] if i < len(invs) else {}
            produce = sum(v for k, v in invi.items()
                          if k not in ("WHEAT", "FERTILIZER") and k not in ANIMALS)
            tgt = min(sheds, key=lambda s: self.dist(units[i], s))
            # Produce sitting in a unit's inventory cannot be sold - only the
            # shed feeds the market - and at end of day it all lands at once,
            # with anything over capacity destroyed. So the fuller the farm
            # gets, the earlier a unit should run its load back to the shed.
            thresh = p["drop_threshold"]
            if self._shed_pressure > p["shed_panic_frac"]:
                thresh = p["drop_threshold_panic"]
            # The episode stops before the final end-of-day drop, so anything
            # still on a unit at the end is destroyed - and only the shed can
            # be sold from. Late on the last day, run everything home.
            if self._liquidate:
                thresh = 1
            if produce >= thresh:
                pri = p["pri_drop"] + produce * 0.5
                if self._liquidate:
                    pri = 1000.0
                logistics.append((pri, i, tgt, ["DROP"]))
            elif animal_need and not any(invi.get(a, 0) for a in animal_need):
                a = next(iter(animal_need))
                k = min(animal_need[a], shed.get(a, 0))
                if k > 0:
                    logistics.append((p["pri_pickup"] + 3, i, tgt, ["PICKUP", a, k]))
                    animal_need.pop(a)
            elif (p["fetch_mode"] < 0.5 and need_wheat
                  and invi.get("WHEAT", 0) <= 0 and wheat_runners > 0):
                wheat_runners -= 1
                # Take a fair share of the feeding, not a flat load. A fixed
                # carry means every extra hand pulls another 8 wheat out of the
                # shed, which then has to be re-bought at a scarcity price.
                # One PICKUP moves n items for one action, so a bigger load is
                # nearly free while each extra shed trip costs the walk there
                # AND back. Top agents run ~260 pickups for ~320 feeds; we were
                # running ~480, burning ~200 actions on logistics.
                share = int(math.ceil(self._unfed / max(1, n))) + 1
                take = max(int(p["wheat_min_carry"]),
                           min(int(p["wheat_carry"]), share * int(p["carry_mult"])))
                logistics.append((p["pri_pickup"], i, tgt, ["PICKUP", "WHEAT", take]))
            elif (p["fetch_mode"] < 0.5 and need_fert
                  and invi.get("FERTILIZER", 0) <= 0 and fert_runners > 0):
                fert_runners -= 1
                logistics.append((p["pri_pickup"] - 40, i, tgt,
                                  ["PICKUP", "FERTILIZER", int(p["fert_carry"])]))

        logistics.sort(reverse=True, key=lambda t: t[0])
        for pri, i, tgt, act in logistics:
            if used[i]:
                continue
            if tuple(units[i]) == tuple(tgt):
                actions[i] = act
            else:
                mv = self.step_towards(units[i], tgt)
                if mv is None:
                    continue
                actions[i] = mv
            used[i] = True

        # -- sticky targeting ------------------------------------------------
        # Re-solving the assignment from scratch every turn makes units thrash:
        # a unit walks halfway to a tile, loses it to a closer unit, turns
        # around. We therefore keep each unit's target until it is done or
        # clearly outclassed.
        by_pos = {}
        for t in tasks:
            q = tuple(t[1])
            if q not in by_pos or t[0] > by_pos[q][0]:
                by_pos[q] = t
        claimed = set()

        def can_do(i, need):
            invi = invs[i] if i < len(invs) else {}
            if need == "WHEAT":
                return invi.get("WHEAT", 0) > 0 or (
                    p["fetch_mode"] >= 0.5 and shed.get("WHEAT", 0) > 0)
            if need == "FERTILIZER":
                return invi.get("FERTILIZER", 0) > 0 or (
                    p["fetch_mode"] >= 0.5 and shed.get("FERTILIZER", 0) > 0)
            if isinstance(need, str) and need in ANIMALS:
                return invi.get(need, 0) > 0
            return True

        def detour(i, need, pos):
            """Extra distance if this unit must swing by the shed first.

            Capping how many units may fetch starves feeding coverage. Instead
            we let the matcher pick the best unit for the job and charge it the
            real cost of collecting what it needs, so exactly the units that
            will actually feed/fertilise make the trip - and nobody tops up
            'just in case' only to have the game drop it again at nightfall.
            """
            if p["fetch_mode"] < 0.5 or need not in ("WHEAT", "FERTILIZER"):
                return 0.0
            invi = invs[i] if i < len(invs) else {}
            if invi.get(need, 0) > 0:
                return 0.0
            sh = min(sheds, key=lambda q: self.dist(units[i], q))
            return (self.dist(units[i], sh) + self.dist(sh, pos)
                    - self.dist(units[i], pos))

        def do_task(i, pos, act, need):
            """Emit the action, routing via the shed when the unit is empty."""
            invi = invs[i] if i < len(invs) else {}
            if (p["fetch_mode"] >= 0.5 and need in ("WHEAT", "FERTILIZER")
                    and invi.get(need, 0) <= 0):
                sh = min(sheds, key=lambda q: self.dist(units[i], q))
                self._target[i] = tuple(pos)      # resume here after collecting
                if tuple(units[i]) == tuple(sh):
                    if need == "WHEAT":
                        take = max(int(p["wheat_min_carry"]),
                                   min(int(p["wheat_carry"]),
                                       int(math.ceil(self._unfed / max(1, n))) + 1))
                    else:
                        take = int(p["fert_carry"])
                    return ["PICKUP", need, take]
                mv = self.step_towards(units[i], sh)
                return mv if mv else ["PASS"]
            if tuple(units[i]) == tuple(pos):
                self._target.pop(i, None)
                return act
            mv = self.step_towards(units[i], pos)
            if mv:
                self._target[i] = tuple(pos)
                return mv
            return None

        for i in range(n):
            if used[i]:
                self._target.pop(i, None)
                continue
            tgt = self._target.get(i)
            if tgt is None or tgt in claimed or tgt not in by_pos:
                continue
            pri, pos, act, need = by_pos[tgt]
            if not can_do(i, need):
                self._target.pop(i, None)
                continue
            claimed.add(tgt)
            if tuple(units[i]) == tuple(pos):
                actions[i] = act
                self._target.pop(i, None)   # done here, pick a new tile next turn
            else:
                mv = self.step_towards(units[i], pos)
                actions[i] = mv if mv else act
            used[i] = True

        # -- global best-pair matching ---------------------------------------
        # Assigning tasks in priority order lets each task grab its nearest
        # unit without regard to what that costs the *other* tasks, so units
        # criss-cross the farm. Repeatedly taking the globally best
        # (unit, task) pair instead is a greedy approximation to the
        # assignment problem and cuts travel substantially.
        if p["match_mode"] >= 0.5:
            cand = sorted(tasks, reverse=True, key=lambda t: t[0])[:int(p["match_cap"])]
            cand = [t for t in cand if tuple(t[1]) not in claimed]
            free = [i for i in range(n) if not used[i]]
            while free and cand:
                bi = bt = None
                bscore = -1e18
                for i in free:
                    ui = units[i]
                    for t in cand:
                        if not can_do(i, t[3]):
                            continue
                        d = self.dist(ui, t[1]) + detour(i, t[3], t[1])
                        sc = t[0] - d * p["dist_penalty"]
                        if sc > bscore:
                            bscore, bi, bt = sc, i, t
                if bi is None:
                    break
                pri, pos, act, need = bt
                q = tuple(pos)
                claimed.add(q)
                a = do_task(bi, pos, act, need)
                if a is not None:
                    actions[bi] = a
                used[bi] = True
                free.remove(bi)
                cand.remove(bt)
            return actions

        # -- round A: critical tasks get the nearest unit, whoever it is ------
        # Losing an animal or a plant is worth crossing the farm for.
        crit = p["critical_pri"]
        ordered = sorted(tasks, reverse=True, key=lambda t: t[0])
        for pri, pos, act, need in ordered:
            if pri < crit or all(used):
                continue
            q = tuple(pos)
            if q in claimed:
                continue
            best, best_score = None, -1e18
            for i in range(n):
                if used[i] or not can_do(i, need):
                    continue
                score = -self.dist(units[i], pos)
                if score > best_score:
                    best_score, best = score, i
            if best is None:
                continue
            i = best
            claimed.add(q)
            if tuple(units[i]) == tuple(pos):
                actions[i] = act
            else:
                mv = self.step_towards(units[i], pos)
                if mv is None:
                    continue
                actions[i] = mv
                self._target[i] = q
            used[i] = True

        # -- round B: every other unit picks its OWN best nearby task --------
        # Unit-centric selection is what keeps travel down: a task-centric loop
        # lets a high-priority tile on the far side of the farm drag a unit
        # across the board, and with 60+ tiles that dominates the action budget.
        free_units = [i for i in range(n) if not used[i]]
        remaining = [t for t in ordered if tuple(t[1]) not in claimed and t[0] < crit]
        for i in free_units:
            best, best_score = None, -1e18
            for t in remaining:
                pri, pos, act, need = t
                q = tuple(pos)
                if q in claimed or not can_do(i, need):
                    continue
                score = pri - self.dist(units[i], pos) * p["dist_penalty"]
                if score > best_score:
                    best_score, best = score, t
            if best is None:
                continue
            pri, pos, act, need = best
            q = tuple(pos)
            claimed.add(q)
            if tuple(units[i]) == tuple(pos):
                actions[i] = act
            else:
                mv = self.step_towards(units[i], pos)
                if mv is None:
                    continue
                actions[i] = mv
                self._target[i] = q
            used[i] = True

        return actions


# --------------------------------------------------------------------------
# Economic forecast / best-response allocation (inlined).
# --------------------------------------------------------------------------
DAYS = 30


def _shop_daily_draw(shops, item):
    """Units/day the unlocked shops consume of `item` (interval 4 -> 6/day)."""
    if item == "FERTILIZER":
        return 0.0
    per = 0.0
    for s in shops:
        prods = SHOPS.get(s, ())
        if item in prods:
            per += 12.0 if len(prods) == 1 else 6.0
    return per


def _centre_daily_draw(item, day):
    if item == "FERTILIZER":
        return 0.0
    mult = 4 if day >= 20 else (2 if day >= 10 else 1)
    return 2.0 * mult          # 24/12 ticks per day


class Forecast:
    """Projected inventory / price path for every product."""

    def __init__(self, obs, me, params):
        self.p = params
        self.day = obs["day"]
        self.me = me
        self.shops = list(obs.get("town", {}).get("unlocked_shops", []) or [])
        self.inv0 = dict(obs["market"]["inventory"])
        farms = obs["farms"]
        self.my_farm = farms[me]
        self.opp_farm = farms[1 - me] if len(farms) > 1 else None

        # demand[item][d] = units the town removes on day d (d is absolute)
        self.demand = {}
        n_unlocked = len(self.shops)
        interval = max(1, int(self.p.get("shop_unlock_interval", 3)))
        remaining = [s for s in SHOPS if s not in self.shops]
        for item in PRODUCTS:
            row = [0.0] * (DAYS + 2)
            base = _shop_daily_draw(self.shops, item)
            # expected contribution of shops that have not unlocked yet
            future = 0.0
            if remaining and item != "FERTILIZER":
                hits = sum((12.0 if len(SHOPS[s]) == 1 else 6.0)
                           for s in remaining if item in SHOPS[s])
                future = hits / len(remaining)
            for d in range(self.day, DAYS + 2):
                extra_unlocks = max(0, (d - self.day) // interval)
                exp_new = min(len(remaining), extra_unlocks)
                row[d] = base + _centre_daily_draw(item, d) + future * exp_new
            self.demand[item] = row

        # supply already committed by each side
        self.opp_supply = self._farm_supply(self.opp_farm)
        self.my_supply = self._farm_supply(self.my_farm)
        # our own shed is sellable stock, already produced
        shed = obs.get("private", {}).get("shed", {}) or {}
        for item, n in shed.items():
            if item in PRODUCTS and n:
                self.my_supply.setdefault(item, [0.0] * (DAYS + 2))
                self.my_supply[item][self.day] += n

    # ------------------------------------------------------------------ supply
    def _farm_supply(self, farm):
        """Units per absolute day each side is committed to producing."""
        out = {item: [0.0] * (DAYS + 2) for item in PRODUCTS}
        if not farm:
            return out
        for row in farm["tiles"]:
            for t in row:
                if not isinstance(t, dict):
                    continue
                kind = t.get("kind")
                if kind == "PLANT":
                    crop = t.get("crop")
                    cd = CROPS.get(crop)
                    if not cd:
                        continue
                    planted = t.get("planted_day", self.day)
                    held = t.get("yield_units", 0) or 0
                    if held:
                        out[crop][min(DAYS, self.day + 1)] += held
                    if cd["ongoing"]:
                        fy, iv, my = cd["first_yield_day"], cd["interval"], cd["max_yield"]
                        for k in range(my):
                            d = planted + fy + k * iv
                            if d > self.day and d <= DAYS:
                                out[crop][d] += 1.0
                    else:
                        d = planted + cd["max_yield_day"]
                        if d > self.day and d <= DAYS:
                            out[crop][d] += cd["max_yield"] * 0.7
                elif kind in ("COOP", "PASTURE") and t.get("animal"):
                    a = t["animal"]
                    ad = ANIMALS[a]
                    prod = ad["product"]
                    placed = t.get("placed_day", self.day)
                    held = t.get("yield_units", 0) or 0
                    if held:
                        out[prod][min(DAYS, self.day + 1)] += held
                    d = placed + ad["first_yield_day"]
                    while d <= DAYS:
                        if d > self.day:
                            out[prod][d] += min(ad["max_held"], 1 + ad["interval"]) * 0.8
                        d += ad["interval"]
                    # every surviving animal also yields a fertiliser a day
                    for dd in range(self.day + 1, DAYS + 1):
                        out["FERTILIZER"][dd] += 1.0
        return out

    def disc(self, d):
        """Shadow price of capital.

        A coin earned on day 12 is worth far more than the same coin on day 28,
        because early cash buys seeds, livestock and land that compound for the
        rest of the season. Once there is no time left to reinvest, cash is
        just cash and the premium disappears. Ignoring this is why we planted 6
        melons where the strong agents planted 26: their day-12 melon harvest
        is the war chest that funds the whole second half.
        """
        horizon = self.p.get("reinvest_until", 18.0)
        rate = self.p.get("capital_rate", 0.10)
        return (1.0 + rate) ** max(0.0, horizon - d)

    # ------------------------------------------------------------------ prices
    def price_path(self, item, extra=None):
        """Predicted price of `item` on each day, given both sides' supply.

        `extra` is an optional per-day array of additional supply from tiles we
        are considering planting - that is what makes the marginal valuation
        self-consistent (our own extra output depresses our own future price).
        """
        inv = self.inv0[item]
        dem = self.demand[item]
        mine = self.my_supply[item]
        opp = self.opp_supply[item]
        out = [0.0] * (DAYS + 2)
        for d in range(self.day, DAYS + 2):
            add = mine[d] + opp[d] + (extra[d] if extra else 0.0)
            inv += add - dem[d]
            if inv < 0:
                inv = 0
            out[d] = market_price(item, int(inv))
        return out

    # ----------------------------------------------------------- tile valuation
    def value_crop(self, crop, days_left, extra=None):
        """(coins_per_tile_day, units, days_used, schedule) for planting now."""
        cd = CROPS[crop]
        day = self.day
        sched = []
        if cd["ongoing"]:
            fy, iv, my = cd["first_yield_day"], cd["interval"], cd["max_yield"]
            fert = 2 if self._fert_available else 1
            for k in range(my):
                d = day + fy + k * iv
                if d <= DAYS - 1:
                    sched.append((d, fert))
            if not sched:
                return -1e9, 0, 1, []
            need = sched[-1][0] - day
        else:
            myd, fyd = cd["max_yield_day"], cd["first_yield_day"]
            ws = (myd + 1) // 2
            harvest = min(day + myd, DAYS - 1)
            age = harvest - day
            if age < fyd:
                return -1e9, 0, 1, []
            units = min(cd["max_yield"], 1 + max(0, age - ws + 1))
            sched = [(harvest, units)]
            need = max(1, age)
        prices = self.price_path(crop, extra)
        rev = sum(u * prices[min(d, DAYS + 1)] * self.disc(d) for d, u in sched)
        units = sum(u for _, u in sched)
        net = rev - cd["seed"]
        return net / max(1.0, need), units, need, sched

    def value_animal(self, animal, days_left, extra=None):
        ad = ANIMALS[animal]
        day = self.day
        prod = ad["product"]
        sched = []
        d = day + ad["first_yield_day"]
        per = min(ad["max_held"], 1 + ad["interval"])
        while d <= DAYS - 1:
            sched.append((d, per))
            d += ad["interval"]
        if not sched:
            return -1e9, 0, 1, [], []
        prices = self.price_path(prod, extra)
        rev = sum(u * prices[min(dd, DAYS + 1)] * self.disc(dd) for dd, u in sched)
        # a fertiliser a day, sold at the projected fertiliser price
        fprices = self.price_path("FERTILIZER")
        fsched = [(dd, 1.0) for dd in range(day + 1, DAYS)]
        frev = sum(fprices[min(dd, DAYS + 1)] * self.disc(dd) for dd, _ in fsched)
        feed = (DAYS - day) * self.p.get("feed_cost", 22.0)
        net = rev + frev * self.p.get("fert_credit", 1.0) - ad["cost"] - feed
        horizon = max(1.0, DAYS - day)
        return net / horizon, sum(u for _, u in sched), horizon, sched, fsched

    # ------------------------------------------------------------- allocation
    def allocate(self, n_slots, budget, capacity, c, days_left):
        """Greedy best response: fill tiles with the highest marginal value,
        charging each choice against our own projected supply as we go."""
        self._fert_available = (c["n_animals"] > 0)
        extra = {item: [0.0] * (DAYS + 2) for item in PRODUCTS}
        pending = {}
        plan = []
        caps = {
            "GOOSE": self.p["max_goose"], "COW": self.p["max_cow"],
            "SHEEP": self.p["max_sheep"], "MELON": self.p["max_melon"],
            "STRAWBERRY": self.p["max_strawberry"], "TOMATO": self.p["max_tomato"],
            "WHEAT": self.p["max_wheat"], "CARROT": self.p["max_carrot"],
        }
        for _ in range(n_slots):
            best, best_v, best_kind = None, 0.0, None
            for a in ANIMALS:
                have = c["animals"][a] + pending.get(a, 0)
                if have >= caps.get(a, 0) or ANIMALS[a]["cost"] > budget:
                    continue
                v = self.value_animal(a, days_left, extra[ANIMALS[a]["product"]])[0]
                v *= self.p.get("w_" + a.lower(), 1.0)
                if v > best_v:
                    best_v, best, best_kind = v, a, "ANIMAL"
            for cr in CROPS:
                have = c["crops"][cr] + pending.get(cr, 0)
                if have >= caps.get(cr, 0) or CROPS[cr]["seed"] > budget:
                    continue
                v = self.value_crop(cr, days_left, extra[cr])[0]
                v *= self.p.get("w_" + ("straw" if cr == "STRAWBERRY" else cr.lower()), 1.0)
                if v > best_v:
                    best_v, best, best_kind = v, cr, "CROP"
            if best is None:
                break
            cost = ANIMALS[best]["cost"] if best_kind == "ANIMAL" else CROPS[best]["seed"]
            acts = (self.p["acts_per_animal"] if best_kind == "ANIMAL"
                    else self.p["acts_per_crop"])
            if cost > budget or acts > capacity:
                break
            budget -= cost
            capacity -= acts
            pending[best] = pending.get(best, 0) + 1
            # book this tile's output against our own projected supply
            if best_kind == "CROP":
                _, _, _, sched = self.value_crop(best, days_left, extra[best])
                for d, u in sched:
                    extra[best][min(d, DAYS + 1)] += u
            else:
                _, _, _, sched, fsched = self.value_animal(best, days_left,
                                                           extra[ANIMALS[best]["product"]])
                prod = ANIMALS[best]["product"]
                for d, u in sched:
                    extra[prod][min(d, DAYS + 1)] += u
                for d, u in fsched:
                    extra["FERTILIZER"][min(d, DAYS + 1)] += u
            plan.append((best_kind, best))
        return plan


# --------------------------------------------------------------------------
# Kaggle entry point.
# --------------------------------------------------------------------------
TUNED_PARAMS = {}

_AGENT = Agent(TUNED_PARAMS)
_SAFE = {"farmer": ["PASS"], "hands": [], "market": []}
# Per-turn budget is 1s with a 60s episode-wide overage bank; we stay ~1000x
# under it, but bail out defensively rather than risk a TIMEOUT forfeit.
_TURN_BUDGET = 0.45


def agent(obs, config=None):
    try:
        t0 = time.perf_counter()
        out = _AGENT.act(obs)
        if time.perf_counter() - t0 > _TURN_BUDGET:
            _AGENT.p["match_cap"] = max(20.0, _AGENT.p["match_cap"] * 0.5)
        return out
    except Exception:
        try:
            n = len(obs["farms"][obs["player"]]["hands"])
        except Exception:
            n = 0
        return {"farmer": ["PASS"], "hands": [["PASS"]] * n, "market": []}


In [ ]:
# Kaggle also accepts a main.py; write both so either works.
import shutil
shutil.copy('submission.py', 'main.py')
print('wrote submission.py and main.py')


In [ ]:
import time
from kaggle_environments import make

# 1. Beats the built-in baseline from BOTH seats (seat 0 has a small edge in
#    this game, so checking only one seat is misleading).
for seat in (0, 1):
    agents = ["submission.py", "starter"] if seat == 0 else ["starter", "submission.py"]
    env = make("kaggriculture", debug=True)
    env.run(agents)
    final = env.steps[-1]
    me, opp = final[seat], final[1 - seat]
    print(f"seat {seat}: {me['status']:>5}  us={me['reward']:>10,.0f}  starter={opp['reward']:>9,.0f}")


In [ ]:
# 2. Self-play — this is exactly the validation episode Kaggle runs on upload.
env = make("kaggriculture", debug=True)
env.run(["submission.py", "submission.py"])
print([(s["status"], f"{s['reward']:,.0f}") for s in env.steps[-1]])


In [ ]:
# 3. Latency. The env allows 1s per turn with a 60s episode-wide overage
#    bank; blowing either forfeits the game. We time the agent on real
#    observations recorded from the episode above.
import submission

obs_list = [env.steps[i][0].observation for i in range(len(env.steps))]
worst = total = 0.0
for obs in obs_list:
    t0 = time.perf_counter()
    submission.agent(obs)
    d = time.perf_counter() - t0
    worst = max(worst, d)
    total += d
print(f"turns timed : {len(obs_list)}")
print(f"worst turn  : {worst*1000:7.2f} ms   (limit 1000 ms)")
print(f"total       : {total:7.2f} s    (60 s overage bank)")


In [ ]:
# Watch a game against the baseline.
env = make("kaggriculture", debug=True)
env.run(["submission.py", "starter"])
env.render(mode="ipython", width=900, height=700)
